# 🌾 SAQYA AI (ساقية) — Teaching the embedding model Egyptian Arabic

Farmers ask the way they speak: *"إمتى أزرع الأوطة؟"*, *"الرز عندي فيه البكانا، أعالجه إزاي؟"*. The bulletins are written
in standard Arabic (and some in English). `BAAI/bge-m3` reads standard Arabic very well, but colloquial words and the
names Egyptian farmers use for crops (الأوطة، الدرة، البتنجان، المانجا، البرتقان) pull its question vectors away from the
passages that answer them. The app already rewrites the commonest Egyptian words into standard Arabic
(`backend/app/services/dialect.py`), but a word list can never cover everything a farmer says.

This notebook teaches **the same bge-m3 model** Egyptian Arabic by **distillation** — the standard way to add a
language to an embedding model without damaging the languages it already knows — **without any AI service and
without labelling anything by hand**:

1. **Egyptian ↔ standard-Arabic pairs.** Egyptian farming questions, each written with its standard-Arabic twin
   (*"إمتى أزرع {crop}؟"* = *"متى أزرع {crop}؟"*), are filled with every crop and topic the documents cover. The
   documents' own Arabic headings become Egyptian requests (*"اشرحلي ري البنجر"* = *"اشرح ري بنجر السكر"*), and
   their sentences are repeated with the crop names farmers use (الأوطة = الطماطم).
2. **The original model is the teacher.** For each pair, the model learns to put the Egyptian text where the
   original bge-m3 puts the standard-Arabic text — a place that is already known to find the right passages.
3. **Nothing else may move.** Standard-Arabic and English questions, headings and thousands of passages are
   trained to keep the vectors the original model gives them, so the index and the other languages stay as good
   as before. Only the top `TRAIN_LAYERS` of the 24 layers learn.
4. **A test before and after** on `data/eval_questions_egyptian.json` (55 questions written by hand in Egyptian
   Arabic — the same facts as q01–q55 in `eval_questions.json`) and on the original Arabic and English questions.
   The model is saved **only if it is better**: the Egyptian questions rank their answers higher, and no test set
   finds fewer answers or ranks them lower.

The result goes to `models/bge-m3-egyptian/`. `rag_pipeline.ipynb` uses it automatically when that folder exists, it
is written into `rag_config.json`, and the backend loads it from there — nothing else to change.

**Order:** `clean_rag_pdfs_no_ai.ipynb` → `data_check.ipynb` → **this notebook** → `rag_pipeline.ipynb` (embeds the
documents with the new model) → restart the backend.

**Time and disk:** with an NVIDIA GPU about 15–25 minutes; on a CPU the settings shrink automatically and it takes
1–3 hours. The first run downloads bge-m3 (≈2.2 GB) if it is not cached yet; the new model takes ≈2.2 GB in `models/`
(not committed to git).

## 1. Settings
Everything you may want to change is here. `DEVICE = "auto"` picks the GPU when PyTorch can see one.

In [1]:
import json
import math
import random
import re
import shutil
import sys
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

pd.set_option("display.max_colwidth", 120)

# Paths work whether Jupyter was started in the project root or in notebooks/
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "backend"))   # reuse the backend's glossary, dialect rules and chunker

BASE_MODEL = "BAAI/bge-m3"                           # the model the pipeline used until now
OUTPUT_DIR = PROJECT_ROOT / "models" / "bge-m3-egyptian"
CLEAN_DIR = PROJECT_ROOT / "data" / "clean"
RAW_DIR = PROJECT_ROOT / "data" / "raw"
SUPPORTED = {".pdf", ".docx", ".txt", ".md"}
DOCS_DIR = CLEAN_DIR if any(p.suffix.lower() in SUPPORTED for p in CLEAN_DIR.glob("*")) else RAW_DIR
VECTOR_STORE_DIR = PROJECT_ROOT / "backend" / "data" / "vector_store"
EVAL_EGYPTIAN_FILE = PROJECT_ROOT / "data" / "eval_questions_egyptian.json"
EVAL_FILE = PROJECT_ROOT / "data" / "eval_questions.json"
RESULTS_DIR = PROJECT_ROOT / "notebooks" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ---- training settings
TRAIN_LAYERS = 6            # top transformer layers that learn (bge-m3 has 24); the rest stay as they are
EPOCHS = "auto"             # passes over the examples: 2 on a GPU, 1 on a CPU
BATCH_SIZE = 16             # lower to 8 if the GPU runs out of memory
LEARNING_RATE = 2e-5
WARMUP = 0.1                # share of the steps with a rising learning rate
MAX_SEQ_LENGTH = 256        # tokens per text while training (the passages are ~200-250 tokens)
MAX_EXAMPLES = "auto"       # training examples: 14000 on a GPU, 3500 on a CPU
MAX_HEADINGS = 2500         # Arabic section headings turned into Egyptian requests
MAX_SENTENCES = 1500        # document sentences repeated with Egyptian crop names
PASSAGE_ANCHORS = "auto"    # passages that must keep their vectors: 3000 on a GPU, 600 on a CPU
EVAL_CORPUS = "auto"        # passages searched in the tests: all on a GPU, 1500 (+ all answer passages) on a CPU
TOP_K = 6                   # passages the app gives the LLM
DEVICE = "auto"             # "cuda", "mps", "cpu" or "auto"
KEEP_ONLY_IF_BETTER = True  # save only a model that improves Egyptian questions and loses nothing elsewhere
SEED = 42

if DEVICE == "auto":
    DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
ON_GPU = DEVICE != "cpu"
if EPOCHS == "auto":
    EPOCHS = 2 if ON_GPU else 1
if MAX_EXAMPLES == "auto":
    MAX_EXAMPLES = 14000 if ON_GPU else 3500
if PASSAGE_ANCHORS == "auto":
    PASSAGE_ANCHORS = 3000 if ON_GPU else 600
if EVAL_CORPUS == "auto":
    EVAL_CORPUS = None if ON_GPU else 1500
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)



def rel(path: Path) -> str:
    """A path as shown to you: relative to the project when it is inside it."""
    try:
        return path.relative_to(PROJECT_ROOT).as_posix()
    except ValueError:
        return str(path)


from app.services.glossary import GLOSSARY, count_terms, document_topic   # noqa: E402
from app.utils.text import detect_language, normalize_for_match             # noqa: E402

print("Project root :", PROJECT_ROOT)
print("Documents    :", DOCS_DIR)
print("Base model   :", BASE_MODEL, "->", rel(OUTPUT_DIR))
print("Device       :", DEVICE, f"({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else "")
if not ON_GPU:
    print("⚠️ No GPU found: training runs on the CPU with smaller settings (1-3 hours). With an NVIDIA GPU, install the\n"
          "   CUDA build of PyTorch (pytorch.org -> Get Started) and restart the kernel.")

Project root : D:\agri-rag-assistant
Documents    : D:\agri-rag-assistant\data\clean
Base model   : BAAI/bge-m3 -> models/bge-m3-egyptian
Device       : cuda (NVIDIA GeForce RTX 4060)


## 2. The passages
If the vector store was built from the documents as they are now, its passages are used (exactly what the app
searches) — and when it was built with `BASE_MODEL`, its vectors also give the "before" test for free.
Otherwise the documents in `data/clean/` are read and chunked here with the pipeline's own code
(`backend/app/services/extraction.py`, the same code the app uses for uploaded documents) and cached in
`notebooks/results/` for the next run.

In [2]:
doc_files = sorted(p for p in DOCS_DIR.iterdir() if p.suffix.lower() in SUPPORTED)
assert doc_files, f"No documents in {DOCS_DIR}"
newest_doc = max(p.stat().st_mtime for p in doc_files)


def current_store_config() -> dict | None:
    """rag_config.json, if the vector store was built from the documents as they are now."""
    cfg_file = VECTOR_STORE_DIR / "rag_config.json"
    if not cfg_file.exists():
        return None
    cfg = json.loads(cfg_file.read_text(encoding="utf-8"))
    try:
        built = datetime.fromisoformat(cfg.get("built_at", "")).timestamp()
    except ValueError:
        return None
    same_documents = set(cfg.get("documents", [])) <= {p.name for p in doc_files}
    return cfg if built > newest_doc and same_documents and cfg.get("num_chunks") else None


store_cfg = current_store_config()
chunks, base_vectors = None, None   # base_vectors: the store's vectors when it was built with BASE_MODEL
if store_cfg:
    try:
        import chromadb

        collection = chromadb.PersistentClient(path=str(VECTOR_STORE_DIR)).get_collection(
            store_cfg["collection_name"], embedding_function=None)
        data = collection.get(include=["documents", "metadatas", "embeddings"])
        chunks = [{"id": i, "text": t, "metadata": m or {}}
                  for i, t, m in zip(data["ids"], data["documents"], data["metadatas"])]
        if store_cfg.get("embedding_model") == BASE_MODEL:
            base_vectors = np.asarray(data["embeddings"], dtype=np.float32)
        print(f"Using the vector store: {len(chunks)} passages "
              f"(built {store_cfg['built_at']} with {store_cfg['embedding_model']})")
    except Exception as exc:  # noqa: BLE001 - fall back to reading the documents
        print(f"Could not read the vector store ({exc}); reading the documents instead")
        chunks, base_vectors = None, None
if chunks is None:
    # Read and chunk the documents with the pipeline's own code (the same code the app uses for uploads)
    from app.services.extraction import ExtractionError, chunk_document, read_document

    cache = RESULTS_DIR / "egyptian_finetune_chunks.json"
    signature = [[p.name, p.stat().st_size, int(p.stat().st_mtime)] for p in doc_files]
    cached = json.loads(cache.read_text(encoding="utf-8")) if cache.exists() else {}
    if cached.get("signature") == signature:
        chunks = cached["chunks"]
        print(f"Reusing {len(chunks)} passages chunked earlier ({rel(cache)})")
    else:
        print(f"The vector store is missing or older than the documents: reading {len(doc_files)} documents "
              "(a few minutes)...")
        chunks, t0 = [], time.time()
        for n, path in enumerate(doc_files, 1):
            try:
                doc = read_document(path)
            except ExtractionError as exc:
                print(f"  skipped {path.name}: {exc}")
                continue
            doc_chunks = chunk_document(doc)
            term = document_topic([c["text"] for c in doc_chunks], path.name)
            for c in doc_chunks:
                c["id"] = f"{path.stem}-{c['id']}"
                c["metadata"]["topic"] = f"{term.ar} / {term.en}" if term else ""
            chunks += doc_chunks
            if n % 10 == 0 or n == len(doc_files):
                print(f"  {n}/{len(doc_files)} documents, {len(chunks)} passages ({time.time() - t0:.0f}s)")
        cache.write_text(json.dumps({"signature": signature, "chunks": chunks}, ensure_ascii=False), encoding="utf-8")

langs = Counter(c["metadata"].get("lang") or detect_language(c["text"]) for c in chunks)
print(f"{len(chunks)} passages from {len({c['metadata'].get('source') for c in chunks})} documents | languages: {dict(langs)}")

Reusing 9341 passages chunked earlier (notebooks/results/egyptian_finetune_chunks.json)
9341 passages from 53 documents | languages: {'en': 3540, 'ar': 5801}


## 3. Training examples — built from your documents, no AI
**Crops:** the glossary's crops (`backend/app/services/glossary.py`, the names used in the bulletins) plus what
farmers in Egypt call them. **Topics:** a heading that matches a topic's Arabic words or English pattern tells which
crop/topic questions the documents can answer. Each topic has Egyptian questions with their standard-Arabic twins,
and standard-Arabic / English questions the model must keep answering as before. Extend the lists freely — more
pairs and names mean a model that understands more ways of asking. Arabic words are matched after normalisation
(أ/إ/آ → ا, ة → ه, ى → ي).

In [3]:
from app.services.dialect import to_msa

# ---- crops: the glossary's standard names (as in the bulletins) + what farmers in Egypt call them
CROP_TERMS = {t.en: t for t in GLOSSARY if t.kind == "crop"}
EGYPTIAN_NAMES = {
    "tomato": ["الأوطة", "القوطة", "الأوطة"],
    "maize": ["الدرة", "الدرة الشامي"],
    "sorghum": ["الدرة الرفيعة", "الدرة العويجة"],
    "rice": ["الرز"],
    "sugar beet": ["البنجر"],
    "sugarcane": ["القصب"],
    "faba bean": ["الفول"],
    "sunflower": ["عباد الشمس"],
    "eggplant": ["البتنجان", "البدنجان"],
    "zucchini": ["الكوسا"],
    "cantaloupe": ["الشمام"],
    "mango": ["المانجا", "المنجة"],
    "citrus": ["البرتقان", "البرتقال", "اليوسفي"],
    "date palm": ["النخل", "البلح"],
    "clover": ["البرسيم"],
}


def crop_name(en: str, style: str, rng: random.Random) -> str:
    term = CROP_TERMS[en]
    if style == "eng":
        return term.en
    if style == "egy":
        # farmers use the colloquial name most of the time, the standard one sometimes
        names = EGYPTIAN_NAMES.get(en, [])
        return rng.choice(names) if names and rng.random() < 0.8 else term.ar
    return term.ar


# ---- topics: a section heading that matches `ar` / `en` gets the topic's questions. Order matters: first match wins.
# Arabic patterns are matched on normalised text (أ/إ/آ -> ا, ة -> ه, ى -> ي), whole words.
# egy: (Egyptian question, the same question in standard Arabic) — {c} is the crop. any: the same, for passages
# that name no crop. msa / eng: standard-Arabic and English questions the model must keep answering as before.
TOPICS = [
    ("seed treatment", dict(
        ar="معامله التقاوي|معامله البذور|معامله البذره|تطهير التقاوي|تطهير البذور|التلقيح البكتيري",
        en=r"seed (?:treatment|dressing)|inoculat",
        egy=[("أعامل تقاوي {c} بإيه قبل ما أزرع؟", "بماذا أعامل تقاوي {c} قبل الزراعة؟"),
             ("أطهر بذرة {c} إزاي؟", "كيف أطهر بذور {c}؟"),
             ("بيقولوا ألقح تقاوي {c}، ده بيفيد بإيه؟", "ما فائدة تلقيح تقاوي {c}؟")],
        msa=["كيف تعامل تقاوي {c} قبل الزراعة؟"],
        eng=["How should {c} seed be treated before sowing?"],
    )),
    ("seed rate", dict(
        ar="معدل التقاوي|كميه التقاوي|معدل البذار|كميه البذور|كميه التقاوي اللازمه|معدل الزراعه|التقاوي",
        en=r"seed(?:ing)? rates?|seed quantit|amount of seed",
        egy=[("الفدان عايز كام كيلو تقاوي {c}؟", "كم كيلو من تقاوي {c} يحتاج الفدان؟"),
             ("أحط كام كيلو بذرة {c} في الفدان؟", "كم كيلو من بذور {c} أضع في الفدان؟"),
             ("تقاوي {c} للفدان قد إيه؟", "ما كمية تقاوي {c} للفدان؟")],
        msa=["ما معدل تقاوي {c} للفدان؟", "كم كيلو تقاوي يحتاج فدان {c}؟"],
        eng=["What is the seed rate for {c} per feddan?", "How much {c} seed is needed per feddan?"],
    )),
    ("planting date", dict(
        ar="ميعاد|مواعيد|موعد|العروه|العروات|عروه",
        en=r"planting (?:date|time|season)|sowing (?:date|time)|time of (?:planting|sowing)",
        egy=[("إمتى أزرع {c}؟", "متى أزرع {c}؟"),
             ("{c} بيتزرع إمتى؟", "متى يزرع {c}؟"),
             ("إيه أحسن ميعاد أزرع فيه {c}؟", "ما أفضل موعد لزراعة {c}؟"),
             ("ميعاد زراعة {c} إمتى يا باشمهندس؟", "ما ميعاد زراعة {c}؟")],
        msa=["ما هو الميعاد المناسب لزراعة {c}؟", "متى يزرع {c}؟"],
        eng=["When should {c} be planted?", "What is the best planting date for {c}?"],
        any=[("إمتى أحسن وقت للزراعة؟", "ما أفضل وقت للزراعة؟")],
    )),
    ("nutrient deficiency", dict(
        ar="نقص|اعراض نقص|العناصر الصغري",
        en=r"deficien",
        egy=[("أعرف إزاي إن {c} ناقصه عنصر؟", "كيف أعرف أن {c} يعاني من نقص عنصر غذائي؟"),
             ("ورق {c} بيصفر، ناقصه إيه؟", "أوراق {c} تصفر، فما العنصر الناقص؟"),
             ("إيه علامات نقص الزنك والحديد في {c}؟", "ما أعراض نقص الزنك والحديد في {c}؟")],
        msa=["ما أعراض نقص العناصر الغذائية في {c}؟"],
        eng=["What are the symptoms of nutrient deficiency in {c}?"],
        any=[("الورق بيصفر، الزرع ناقصه إيه؟", "الأوراق تصفر، فما العنصر الذي ينقص النبات؟")],
    )),
    ("irrigation", dict(
        ar="الري|ري|الريه|الريات|مياه الري|السقي|الاحتياجات المائيه|الاحتياج المائي|الصرف",
        en=r"irrigat|watering|water requirement",
        egy=[("أسقي {c} إزاي؟", "كيف أروي {c}؟"),
             ("{c} عايز مية كل قد إيه؟", "كل كم يوم يحتاج {c} إلى الري؟"),
             ("إمتى أسقي {c}؟", "متى أروي {c}؟"),
             ("أروي {c} كام مرة في الموسم؟", "كم رية يحتاج {c} في الموسم؟")],
        msa=["كيف يتم ري {c}؟", "ما عدد ريات {c} والفترة بينها؟"],
        eng=["How should {c} be irrigated?", "How often should {c} be watered?"],
        any=[("أسقي الأرض كل قد إيه؟", "كل كم يوم أروي الأرض؟"),
             ("أعرف إزاي إن الزرع عايز مية؟", "كيف أعرف أن النبات يحتاج إلى الري؟")],
    )),
    ("fertilizer", dict(
        ar="التسميد|تسميد|السماد|سماد|الاسمده|اسمده",
        en=r"fertili[sz]|manure",
        egy=[("أسمد {c} بإيه وإمتى؟", "بماذا ومتى أسمد {c}؟"),
             ("{c} محتاج سماد قد إيه؟", "ما كمية السماد التي يحتاجها {c}؟"),
             ("أحط الكيماوي إمتى في {c}؟", "متى أضيف السماد الكيماوي إلى {c}؟"),
             ("إيه السماد اللي {c} عايزه؟", "ما السماد الذي يحتاجه {c}؟")],
        msa=["ما برنامج تسميد {c}؟", "ما كميات الأسمدة اللازمة لزراعة {c}؟"],
        eng=["How should {c} be fertilized?", "What fertilizer does {c} need?"],
        any=[("أحط سباخ قد إيه في الفدان؟", "ما كمية السماد البلدي للفدان؟"), ("أسمد الأرض إزاي؟", "كيف أسمد الأرض؟")],
    )),
    ("weeds", dict(
        ar="الحشائش|حشائش|العزيق|عزيق|الحشيش",
        en=r"weeds?|weeding|hoeing",
        egy=[("أخلص من الحشيش في {c} إزاي؟", "كيف أتخلص من الحشائش في {c}؟"),
             ("الحشيش مالي غيط {c}، أعمل إيه؟", "الحشائش تملأ حقل {c}، فماذا أفعل؟"),
             ("أعزق {c} إمتى؟", "متى أعزق {c}؟")],
        msa=["كيف تكافح الحشائش في {c}؟"],
        eng=["How can weeds be controlled in {c}?"],
        any=[("أخلص من الحشيش اللي في الغيط إزاي؟", "كيف أتخلص من الحشائش في الحقل؟")],
    )),
    ("pest management", dict(
        ar="المكافحه المتكامله|الاداره المتكامله|المبيدات|مبيدات|مبيد|الرش|مقاومه المبيدات",
        en=r"integrated pest|\bipm\b|pesticide|insecticide|spray",
        egy=[("أرش {c} بإيه وإمتى؟", "بماذا ومتى أرش {c}؟"),
             ("إزاي أقلل رش المبيدات في {c}؟", "كيف أقلل رش المبيدات في {c}؟")],
        msa=["ما برنامج مكافحة الآفات في {c}؟"],
        eng=["How should pesticides be used on {c}?"],
        any=[("أستخدم المبيد إزاي من غير ما أضر نفسي؟", "كيف أستخدم المبيد دون أن أضر نفسي؟"),
             ("إيه طرق مكافحة الآفات من غير كيماوي؟", "ما طرق مكافحة الآفات دون مبيدات كيميائية؟"),
             ("أرش المبيد إزاي صح؟", "ما الطريقة الصحيحة لرش المبيد؟")],
    )),
    ("diseases", dict(
        ar="الامراض|امراض|مرض|العفن|عفن|اعفان|البياض|الصدا|اللفحه|لفحه|الذبول|ذبول|التبقع|تبقع|فيروس|الفيروسيه|"
           "الفطريه|البكتيريه|العفن الطري|التصمغ|البكانا|عفن الجذور",
        en=r"diseases?|blight|\brots?\b|mildew|\brusts?\b|wilt|virus|fung|bacterial|leaf spot|smut",
        egy=[("{c} عندي عيان، أعالجه إزاي؟", "{c} مصاب بمرض، فكيف أعالجه؟"),
             ("إيه الأمراض اللي بتصيب {c}؟", "ما الأمراض التي تصيب {c}؟"),
             ("الورق في {c} عليه بقع، أعمل إيه؟", "على أوراق {c} بقع، فماذا أفعل؟"),
             ("أرش {c} بإيه من العفن؟", "بماذا أرش {c} لمكافحة العفن؟")],
        msa=["ما أهم أمراض {c} وكيف تكافح؟", "كيف أعالج أمراض {c}؟"],
        eng=["What diseases affect {c} and how are they controlled?"],
    )),
    ("pests", dict(
        ar="الافات|افات|الحشرات|حشرات|حشره|دوده|الديدان|ديدان|المن|الذبابه|العنكبوت|الاكاروس|التوتا|سوسه|"
           "النيماتودا|القوارض|الفئران|الثاقبه|البق|التربس|فراشه",
        en=r"pests?|insects?|aphids?|worms?|\bfl(?:y|ies)\b|mites?|borers?|nematodes?|weevils?|tuta|thrips|"
           r"rodents?|moths?|caterpillars?",
        egy=[("{c} عندي فيه حشرات، أعمل إيه؟", "في {c} حشرات، فماذا أفعل؟"),
             ("أقاوم الدود في {c} إزاي؟", "كيف أكافح الديدان في {c}؟"),
             ("إيه الحشرات اللي بتضرب {c}؟", "ما الحشرات التي تصيب {c}؟"),
             ("أرش {c} بإيه عشان الحشرات؟", "بماذا أرش {c} لمكافحة الحشرات؟")],
        msa=["ما أهم آفات {c} وطرق مكافحتها؟", "كيف أكافح الحشرات في {c}؟"],
        eng=["What pests attack {c} and how are they controlled?"],
    )),
    ("storage", dict(
        ar="التخزين|تخزين|التداول|تداول|ما بعد الحصاد|التعبئه|التسويق|الفرز|التدريج|التجفيف|التبريد",
        en=r"storage|post-?harvest|handling|packing|grading|marketing|drying|cooling",
        egy=[("أخزن {c} إزاي عشان ميبوظش؟", "كيف أخزن {c} حتى لا يتلف؟"),
             ("أعمل إيه في {c} بعد الحصاد؟", "ما معاملات {c} بعد الحصاد؟"),
             ("{c} يستحمل التخزين قد إيه؟", "ما مدة تخزين {c}؟")],
        msa=["كيف يخزن {c}؟", "ما معاملات {c} بعد الحصاد؟"],
        eng=["How should {c} be stored after harvest?"],
    )),
    ("harvest", dict(
        ar="الحصاد|حصاد|الجمع|جمع|الجني|جني|التقليع|تقليع|القطف|قطف|النضج|نضج|الكسر|الحش",
        en=r"harvest|maturity|picking|ripen",
        egy=[("إمتى أحصد {c}؟", "متى أحصد {c}؟"),
             ("أعرف إزاي إن {c} استوى؟", "كيف أعرف أن {c} نضج؟"),
             ("{c} بيتجمع إمتى؟", "متى يجمع {c}؟"),
             ("علامات نضج {c} إيه؟", "ما علامات نضج {c}؟")],
        msa=["متى يتم حصاد {c}؟", "ما علامات نضج {c}؟"],
        eng=["When is {c} ready to harvest?", "What are the signs of {c} maturity?"],
    )),
    ("soil", dict(
        ar="تحليل التربه|تحليل الترب|الملوحه|ملوحه|التوصيل الكهربي|حموضه|القلويه|خصوبه التربه",
        en=r"soil (?:test|analysis|sampl)|salinity|saline|electrical conductivity|\bph\b|soil fertility",
        egy=[("الأرض المالحة تنفع لزراعة {c}؟", "هل تصلح الأرض الملحية لزراعة {c}؟"),
             ("{c} يستحمل ملوحة الأرض؟", "هل يتحمل {c} ملوحة التربة؟")],
        msa=["ما مدى تحمل {c} للملوحة؟"],
        eng=["Can {c} tolerate soil salinity?"],
        any=[("أحلل الأرض إزاي وأطلب إيه من المعمل؟", "كيف أحلل التربة وما التحاليل التي أطلبها من المعمل؟"),
             ("الأرض عندي مالحة، أعمل إيه؟", "أرضي ملحية، فماذا أفعل؟"),
             ("أعرف منين إن الأرض مالحة؟", "كيف أعرف أن التربة ملحية؟")],
    )),
    ("land preparation", dict(
        ar="اعداد الارض|تجهيز الارض|خدمه الارض|اعداد التربه|تجهيز التربه|الحرث|حرث|التسويه|الارض المناسبه|"
           "التربه المناسبه|الاراضي المناسبه|الاراضي الملائمه|نوع التربه|الظروف البيئيه|المناخ المناسب|الجو المناسب",
        en=r"land preparation|soil preparation|tillage|plough|plow|suitable soils?|soil type|soil requirement|climat",
        egy=[("أجهز الأرض لزراعة {c} إزاي؟", "كيف أجهز الأرض لزراعة {c}؟"),
             ("{c} بيحب أنهي نوع أرض؟", "ما نوع التربة المناسبة لزراعة {c}؟"),
             ("{c} عايز جو عامل إزاي؟", "ما الظروف الجوية المناسبة لزراعة {c}؟")],
        msa=["كيف يتم إعداد الأرض لزراعة {c}؟", "ما الأراضي والظروف المناسبة لزراعة {c}؟"],
        eng=["How should the land be prepared for {c}?", "What soil and climate suit {c}?"],
    )),
    ("varieties", dict(
        ar="الاصناف|اصناف|الصنف|صنف|الهجن|هجن",
        en=r"variet|cultivar|hybrids?",
        egy=[("إيه أحسن صنف {c} أزرعه؟", "ما أفضل صنف من {c} للزراعة؟"),
             ("أصناف {c} إيه؟", "ما أصناف {c}؟"),
             ("أنهي صنف {c} بيدي محصول أكتر؟", "ما صنف {c} الأعلى إنتاجا؟")],
        msa=["ما أفضل أصناف {c}؟", "ما الأصناف المعتمدة من {c}؟"],
        eng=["Which {c} varieties are recommended?"],
    )),
    ("nursery", dict(
        ar="المشتل|مشتل|المشاتل|الشتلات|شتلات|الشتل|الشتله|شتله|التقسيه|الاقلمه|الفسائل|فسائل",
        en=r"nurser|seedlings?|transplant|hardening|offshoots?",
        egy=[("أعمل مشتل {c} إزاي؟", "كيف أنشئ مشتل {c}؟"),
             ("أشتل {c} إمتى وإزاي؟", "متى وكيف أشتل {c}؟"),
             ("شتلة {c} الكويسة شكلها إيه؟", "ما مواصفات شتلة {c} الجيدة؟"),
             ("أنقل شتلات {c} للأرض إمتى؟", "متى أنقل شتلات {c} إلى الأرض المستديمة؟")],
        msa=["كيف يتم إنتاج شتلات {c}؟", "متى تنقل شتلات {c} إلى الأرض المستديمة؟"],
        eng=["How are {c} seedlings produced and transplanted?"],
    )),
    ("thinning", dict(
        ar="الخف|خف",
        en=r"thinning",
        egy=[("أخف {c} إمتى؟", "متى أخف {c}؟"),
             ("أسيب كام نبات في الجورة بعد خف {c}؟", "كم نباتا أترك في الجورة بعد خف {c}؟")],
        msa=["متى تجرى عملية الخف في {c}؟"],
        eng=["When should {c} be thinned?"],
    )),
    ("pruning", dict(
        ar="التقليم|تقليم|التربيه|التطعيم|تطعيم",
        en=r"prun|training|grafting",
        egy=[("أقلم {c} إزاي وإمتى؟", "كيف ومتى أقلم {c}؟"),
             ("إمتى أقلم شجر {c}؟", "متى أقلم أشجار {c}؟"),
             ("أطعم {c} إزاي؟", "كيف أطعم {c}؟")],
        msa=["كيف يتم تقليم {c}؟"],
        eng=["How and when should {c} be pruned?"],
    )),
    ("pollination", dict(
        ar="التلقيح|تلقيح|التزهير|الازهار|العقد",
        en=r"pollinat|flowering|fruit set",
        egy=[("ألقح {c} إزاي؟", "كيف ألقح {c}؟"),
             ("ليه {c} مش بيعقد؟", "لماذا لا يعقد {c}؟"),
             ("{c} بيزهر إمتى؟", "متى يزهر {c}؟")],
        msa=["كيف يتم تلقيح {c}؟"],
        eng=["How is {c} pollinated?"],
    )),
    ("propagation", dict(
        ar="الاكثار|اكثار",
        en=r"propagat",
        egy=[("أكاثر {c} إزاي؟", "كيف أكثر {c}؟"),
             ("أزرع {c} من البذرة ولا من الشتلة؟", "هل أزرع {c} بالبذور أم بالشتلات؟")],
        msa=["كيف يتم إكثار {c}؟"],
        eng=["How is {c} propagated?"],
    )),
    ("yield and disorders", dict(
        ar="الانتاجيه|المحصول|زياده الانتاج|زياده المحصول|المشاكل|مشاكل|الظواهر|ظواهر|العيوب|عيوب|التشقق|تشقق|"
           "المعاومه|السقوط|تساقط|الفسيولوجيه|لسعه الشمس",
        en=r"yield|disorders?|problems?|crack|alternate bearing|fruit drop|sunburn|physiological",
        egy=[("أزود محصول {c} إزاي؟", "كيف أزيد محصول {c}؟"),
             ("ليه {c} بيطرح قليل؟", "لماذا إنتاج {c} قليل؟"),
             ("ليه ثمار {c} بتقع؟", "لماذا تتساقط ثمار {c}؟"),
             ("{c} فيه مشاكل في الثمار، السبب إيه؟", "ما سبب مشاكل ثمار {c}؟")],
        msa=["كيف تزيد إنتاجية {c}؟", "ما أسباب تساقط ثمار {c}؟", "ما الظواهر الفسيولوجية في {c}؟"],
        eng=["How can {c} yield be increased?", "Why do {c} fruits drop or crack?"],
    )),
    ("planting", dict(
        ar="طرق الزراعه|طريقه الزراعه|طرق زراعه|طريقه زراعه|الكثافه النباتيه|مسافات الزراعه|المسافات|التخطيط|"
           "الزراعه|زراعه",
        en=r"planting|sowing|spacing|plant density|cultivation|growing",
        egy=[("أزرع {c} إزاي؟", "كيف أزرع {c}؟"),
             ("أسيب مسافة قد إيه بين نباتات {c}؟", "ما المسافة بين نباتات {c}؟"),
             ("أزرع {c} على خطوط ولا مصاطب؟", "هل أزرع {c} على خطوط أم على مصاطب؟")],
        msa=["ما طرق زراعة {c}؟", "ما مسافات زراعة {c}؟"],
        eng=["How is {c} planted?", "What spacing should be used for {c}?"],
    )),
]


def _words(pattern: str) -> re.Pattern:
    alts = "|".join(normalize_for_match(w) for w in pattern.split("|"))
    return re.compile(rf"(?<![\w])(?:[وفب]?(?:ال)?)?(?:{alts})(?![\w])")


TOPIC_RES = [(name, _words(t["ar"]), re.compile(t["en"], re.IGNORECASE)) for name, t in TOPICS]
TOPIC = dict(TOPICS)


def topic_mentions(topic: str, text: str) -> int:
    _, ar_re, en_re = next(t for t in TOPIC_RES if t[0] == topic)
    return len(ar_re.findall(normalize_for_match(text))) + len(en_re.findall(text))


def topic_of(heading: str) -> str | None:
    norm = normalize_for_match(heading)
    for name, ar_re, en_re in TOPIC_RES:
        if ar_re.search(norm) or en_re.search(heading):
            return name
    return None


# Egyptian frames around a section heading, with the same request in standard Arabic: "اشرحلي ري البنجر" = "اشرح ري البنجر"
HEADING_FRAMES = [("اشرحلي {h}", "اشرح {h}"), ("عايز أعرف عن {h}", "أريد أن أعرف عن {h}"),
                  ("قولي إيه هي {h}", "ما هي {h}"), ("إيه المطلوب في {h}؟", "ما المطلوب في {h}؟"),
                  ("يعني إيه {h}؟", "ما معنى {h}؟"), ("ممكن تفهمني {h}؟", "هل يمكنك أن تشرح لي {h}؟"),
                  ("عايز أفهم {h}", "أريد أن أفهم {h}")]
NUMBERING_RE = re.compile(r"^(?:[\s.\-–:()]*(?:\d+|[أابجدهوزحطيكل]|اولا|أولا|ثانيا|ثالثا|رابعا|خامسا|سادسا|سابعا|"
                          r"[ivx]+)[\s.\-–:)]+)+", re.IGNORECASE)
TRAILING_RE = re.compile(r"[\s.\-–:(]*\d+[\s.\-–:)]*$")
ARABIC_LETTER_RE = re.compile(r"[ء-ي]")


def clean_heading(h: str) -> str:
    h = NUMBERING_RE.sub("", h.strip())
    h = TRAILING_RE.sub("", h)
    return re.sub(r"\s+", " ", h).strip(" .:-–")


def usable_arabic_heading(h: str) -> bool:
    words = h.split()
    letters = re.sub(r"[\W\d_]", "", h)
    return (1 <= len(words) <= 7 and len(letters) >= 4 and not re.search(r"\d", h)
            and len(ARABIC_LETTER_RE.findall(h)) / max(len(letters), 1) > 0.9)


print(f"{len(CROP_TERMS)} crops ({len(EGYPTIAN_NAMES)} with Egyptian names), {len(TOPICS)} topics, "
      f"{sum(len(t['egy']) + len(t.get('any', [])) for _, t in TOPICS)} Egyptian questions with their standard-Arabic twins")

29 crops (15 with Egyptian names), 22 topics, 83 Egyptian questions with their standard-Arabic twins


Each example is *(text the model reads, text whose original vector it must produce)*. Kinds:
- **egyptian question:** an Egyptian question → its standard-Arabic twin (also with the bulletins' crop name, and
  half of them also after the app's dialect rewrite, because that is the text the app actually searches);
- **egyptian heading:** an Egyptian request around a document heading → the same request in standard Arabic;
- **egyptian crop names:** a document sentence with the farmers' crop name → the original sentence;
- **keep: …** standard-Arabic and English questions, headings and passages → themselves, so they don't move.

Generated questions identical to a test question are dropped, so the test stays fair. The templates and the
hand-written test questions do share everyday phrasing ("إمتى أزرع…؟"), which is the point — but judge the model on
your own questions in the chat app as well.

In [4]:
rng = random.Random(SEED)
CROP_BY_LABEL = {f"{t.ar} / {t.en}": t.en for t in CROP_TERMS.values()}


def chunk_crop(chunk: dict, heading: str) -> str | None:
    """The crop a passage is about: its document's topic, else a crop its heading or text clearly names."""
    label = (chunk["metadata"].get("topic") or "").strip()
    if label in CROP_BY_LABEL:
        return CROP_BY_LABEL[label]
    counts = count_terms(heading + "\n" + chunk["text"], kind="crop")
    if not counts:
        return None
    best = max(counts, key=counts.get)
    if counts[best] >= 2 and counts[best] > sum(counts.values()) - counts[best]:
        return best.en
    return None


def heading_path(chunk: dict) -> list[str]:
    first = chunk["text"].split("\n", 1)[0]
    if " › " in first:
        return [p.strip() for p in first.split(" › ")[1:] if p.strip()]   # without the document title
    section = (chunk["metadata"].get("section") or "").strip()
    return [section] if section else []


# ---- Egyptian crop names inside standard-Arabic text: "تزرع الطماطم في ..." -> "تزرع الأوطة في ..."
_L = "ء-ي"
CROP_SWAPS = []
for en, names in EGYPTIAN_NAMES.items():
    msa = CROP_TERMS[en].ar
    has_article = msa.startswith("ال")
    core = re.escape(msa[2:] if has_article else msa)
    CROP_SWAPS.append((en, has_article, re.compile(rf"(?<![{_L}])(و?)لل{core}(?![{_L}])") if has_article else None,
                       re.compile(rf"(?<![{_L}])(و?[بفك]?)(ال)?{core}(?![{_L}])")))


def egyptianize(text: str) -> str:
    """The same text with the crop names farmers use (one Egyptian name per crop, chosen at random)."""
    for en, has_article, lil_re, word_re in CROP_SWAPS:
        name = rng.choice(EGYPTIAN_NAMES[en])
        name_core = name[2:] if name.startswith("ال") else name
        if lil_re is not None:
            text = lil_re.sub(lambda m: f"{m.group(1)}لل{name_core}", text)
        if has_article:
            text = word_re.sub(lambda m: f"{m.group(1)}{'ال' if m.group(2) else ''}{name_core}", text)
        else:   # "بنجر السكر" -> "البنجر"
            text = word_re.sub(lambda m: f"{m.group(1)}{name}", text)
    return text


# Test questions are never used for training
eval_norm = set()
for f in (EVAL_EGYPTIAN_FILE, EVAL_FILE):
    for q in json.loads(f.read_text(encoding="utf-8")):
        eval_norm |= {normalize_for_match(q["question"]).strip(" ؟?"), normalize_for_match(to_msa(q["question"])).strip(" ؟?")}

examples, seen, dropped_eval = [], set(), 0


def add(student: str, teacher: str, kind: str) -> None:
    """The fine-tuned model must put `student` where the original bge-m3 puts `teacher`."""
    global dropped_eval
    student, teacher = re.sub(r"\s+", " ", student).strip(), re.sub(r"\s+", " ", teacher).strip()
    if student != teacher and normalize_for_match(student).strip(" ؟?") in eval_norm:
        dropped_eval += 1
        return
    if (student, teacher) not in seen:
        seen.add((student, teacher))
        examples.append({"student": student, "teacher": teacher, "kind": kind})


def add_egyptian(egyptian: str, standard: str, kind: str) -> None:
    add(egyptian, standard, kind)
    rewritten = to_msa(egyptian)   # the text the app actually searches, half of the time
    if rewritten not in (egyptian, standard) and rng.random() < 0.5:
        add(rewritten, standard, kind)


# Sections of the documents: which crop and topic they are about
groups: dict[tuple, dict] = {}
for c in chunks:
    path = heading_path(c)
    if path:
        groups.setdefault((c["metadata"].get("source", ""), " › ".join(path)), c)

crop_topics, headings = Counter(), []
for (source, path_text), c in groups.items():
    path = path_text.split(" › ")
    deepest = clean_heading(path[-1])
    topic = topic_of(deepest) or (topic_of(clean_heading(path[0])) if len(path) > 1 else None)
    crop = chunk_crop(c, path_text)
    if topic and crop:
        crop_topics[(crop, topic)] += 1
    lang = c["metadata"].get("lang") or detect_language(c["text"])
    if lang == "ar" and (crop or topic) and usable_arabic_heading(deepest):
        headings.append((deepest, crop))

# 1) Egyptian questions -> the same question in standard Arabic, for every crop/topic the documents cover
for (crop, topic) in crop_topics:
    t = TOPIC[topic]
    names = list(dict.fromkeys([crop_name(crop, "egy", rng), CROP_TERMS[crop].ar]))
    for egy, msa in t["egy"]:
        for name in names:
            add_egyptian(egy.format(c=name), msa.format(c=CROP_TERMS[crop].ar), "egyptian question")
    for msa in t["msa"]:
        add(msa.format(c=CROP_TERMS[crop].ar), msa.format(c=CROP_TERMS[crop].ar), "keep: standard question")
    for eng in t["eng"]:
        add(eng.format(c=CROP_TERMS[crop].en), eng.format(c=CROP_TERMS[crop].en), "keep: english question")
for _, t in TOPICS:
    for egy, msa in t.get("any", []):
        for _ in range(3):
            add_egyptian(egy, msa, "egyptian question")

# 2) Egyptian requests around the documents' own Arabic headings: "اشرحلي ري البنجر" = "اشرح ري بنجر السكر"
rng.shuffle(headings)
for h, crop in headings[:MAX_HEADINGS]:
    h_std = h if not crop or count_terms(h, kind="crop") else f"{h} في {CROP_TERMS[crop].ar}"
    egy_frame, std_frame = rng.choice(HEADING_FRAMES)
    add_egyptian(egy_frame.format(h=egyptianize(h_std)), std_frame.format(h=h_std), "egyptian heading")
    add(h_std, h_std, "keep: heading")

# 3) Sentences from the documents with Egyptian crop names: "تزرع الأوطة في ..." = "تزرع الطماطم في ..."
sentences = []
for c in chunks:
    if (c["metadata"].get("lang") or detect_language(c["text"])) != "ar":
        continue
    for sentence in re.split(r"(?<=[.!؟?])\s+|\n", c["text"]):
        letters = re.sub(r"[\W\d_]", "", sentence)
        arabic = len(re.findall(r"[ء-ي]", sentence)) / max(len(letters), 1)
        if 5 <= len(sentence.split()) <= 30 and " › " not in sentence and "=" not in sentence and arabic > 0.9:
            swapped = egyptianize(sentence)
            if swapped != sentence:
                sentences.append((swapped, sentence))
for swapped, sentence in rng.sample(sentences, min(len(sentences), MAX_SENTENCES)):
    add(swapped, sentence, "egyptian crop names")

# 4) Egyptian crop names alone = the bulletins' names
for en, names in EGYPTIAN_NAMES.items():
    for name in dict.fromkeys(names):
        add(name, CROP_TERMS[en].ar, "egyptian crop names")
        add(f"زراعة {name}", f"زراعة {CROP_TERMS[en].ar}", "egyptian crop names")

# 5) Passages stay where they are: their vectors in the index keep matching standard Arabic and English
for c in rng.sample(chunks, min(len(chunks), PASSAGE_ANCHORS)):
    add(c["text"], c["text"], "keep: passage")

kinds = Counter(e["kind"] for e in examples)
if len(examples) > MAX_EXAMPLES:   # same mix, fewer examples
    examples = rng.sample(examples, MAX_EXAMPLES)
    kinds = Counter(e["kind"] for e in examples)
print(f"{len(crop_topics)} crop/topic combinations and {len(headings)} Arabic headings in the documents")
print(f"{len(examples)} training examples:")
for kind, n in sorted(kinds.items()):
    print(f"  {n:6d}  {kind}")
print(f"Skipped {dropped_eval} generated questions identical to a test question")

PAIRS_FILE = RESULTS_DIR / "egyptian_training_pairs.jsonl"
with PAIRS_FILE.open("w", encoding="utf-8") as f:
    for e in examples:
        f.write(json.dumps(e, ensure_ascii=False) + "\n")
print(f"Saved to {rel(PAIRS_FILE)} — open it to read every example")
sample = [e for e in examples if not e["kind"].startswith("keep")]
pd.DataFrame(rng.sample(sample, min(12, len(sample))))[["kind", "student", "teacher"]].rename(
    columns={"student": "Egyptian (the model learns this…)", "teacher": "…means the same as this"})

350 crop/topic combinations and 2195 Arabic headings in the documents
11641 training examples:
     579  egyptian crop names
    2714  egyptian heading
    2398  egyptian question
     525  keep: english question
    1797  keep: heading
    2991  keep: passage
     637  keep: standard question
Skipped 6 generated questions identical to a test question
Saved to notebooks/results/egyptian_training_pairs.jsonl — open it to read every example


,kind,Egyptian (the model learns this…),…means the same as this
0,egyptian heading,أريد أفهم لسعة الشمس في الموالح,أريد أن أفهم لسعة الشمس في الموالح
1,egyptian crop names,يعتبر ميعاد الزراعة من العوامل المحددة لإنتاج محصول الفول بسبب ارتباط العوامل الجوية )الحرارة- الرطوبة الجوية,يعتبر ميعاد الزراعة من العوامل المحددة لإنتاج محصول الفول البلدي بسبب ارتباط العوامل الجوية )الحرارة- الرطوبة الجوية
2,egyptian question,أعمل إيه في السمسم بعد الحصاد؟,ما معاملات السمسم بعد الحصاد؟
3,egyptian question,الذرة الرفيعة عندي عيان، أعالجه كيف؟,الذرة الرفيعة مصاب بمرض، فكيف أعالجه؟
4,egyptian heading,عايز أفهم المكافحة الكيميائية في البدنجان,أريد أن أفهم المكافحة الكيميائية في الباذنجان
5,egyptian question,أرش البطاطس بإيه من العفن؟,بماذا أرش البطاطس لمكافحة العفن؟
6,egyptian question,أصناف الفلفل إيه؟,ما أصناف الفلفل؟
7,egyptian question,ما الأمراض الذي بتصيب الفول؟,ما الأمراض التي تصيب الفول البلدي؟
8,egyptian heading,إيه المطلوب في زراعة مشتل الفراولة؟,ما المطلوب في زراعة مشتل الفراولة؟
9,egyptian question,ليه البصل بيطرح قليل؟,لماذا إنتاج البصل قليل؟


## 4. Test before training
**Embeddings only** (no keyword search, no translation), so the numbers show the model itself. For each question:
is a passage with the expected answer (same rule as `rag_pipeline.ipynb`: the expected document and one of the
expected keywords) among the top 6 (**hit@6**), and how high is the first one (**MRR@10**: 1 for rank 1, ½ for
rank 2, …, 0 when not in the top 10). Four test sets:
- **Egyptian, as typed** — the hand-written Egyptian questions;
- **Egyptian, after the app's dialect rewrite** — what the app actually searches (`to_msa()`);
- **Standard Arabic** and **English** — the original questions, to check nothing is lost.

The full app does better than these numbers (it adds BM25 keyword search and the Arabic↔English translation).

In [5]:
from sentence_transformers import SentenceTransformer

from app.services.grounding import contains_keyword

model = SentenceTransformer(BASE_MODEL, device=DEVICE)
FULL_MAX_LENGTH = model.max_seq_length
ENCODE_BATCH = 32 if ON_GPU else 8

egyptian_questions = json.loads(EVAL_EGYPTIAN_FILE.read_text(encoding="utf-8"))
in_domain = [q for q in json.loads(EVAL_FILE.read_text(encoding="utf-8")) if q["type"] == "in_domain"]
TEST_SETS = {
    "Egyptian, as typed": [(q, q["question"]) for q in egyptian_questions],
    "Egyptian, after the app's dialect rewrite": [(q, to_msa(q["question"])) for q in egyptian_questions],
    "Standard Arabic (original questions)": [(q, q["question"]) for q in in_domain if detect_language(q["question"]) == "ar"],
    "English": [(q, q["question"]) for q in in_domain if detect_language(q["question"]) == "en"],
}


def is_relevant(q: dict, chunk: dict) -> bool:
    """Same rule as rag_pipeline.ipynb: from the expected document and containing an expected keyword."""
    source = q.get("expected_source", "").lower()
    return source in chunk["metadata"].get("source", "").lower() and contains_keyword(chunk["text"], q["expected_keywords"])


# The test corpus: every passage that answers a test question + (on a CPU) a random sample of the others
questions_by_id = {q["id"]: q for pairs_ in TEST_SETS.values() for q, _ in pairs_}
answers = {qid: {i for i, c in enumerate(chunks) if is_relevant(q, c)} for qid, q in questions_by_id.items()}
answer_idx = set().union(*answers.values())
others = [i for i in range(len(chunks)) if i not in answer_idx]
if EVAL_CORPUS and len(others) > EVAL_CORPUS:
    others = random.Random(SEED).sample(others, EVAL_CORPUS)
eval_idx = sorted(answer_idx | set(others))
position = {i: n for n, i in enumerate(eval_idx)}
eval_answers = {qid: {position[i] for i in idx} for qid, idx in answers.items()}
missing = sorted(qid for qid, idx in eval_answers.items() if not idx)
print(f"Test corpus: {len(eval_idx)} passages ({len(answer_idx)} of them answer a test question)")
if missing:
    print(f"No passage contains the expected answer for {len(missing)} questions (left out): {', '.join(missing)}")


def embed_corpus() -> np.ndarray:
    return model.encode([chunks[i]["text"] for i in eval_idx], batch_size=ENCODE_BATCH, normalize_embeddings=True,
                        convert_to_numpy=True, show_progress_bar=True)


def evaluate(corpus_vectors: np.ndarray) -> tuple[pd.DataFrame, dict]:
    """Embeddings-only search (no BM25, no translation): hit@TOP_K and MRR@10 per test set, and every rank."""
    rows, ranks = [], {}
    for name, items in TEST_SETS.items():
        items = [(q, text) for q, text in items if eval_answers[q["id"]]]
        vectors = model.encode([text for _, text in items], batch_size=ENCODE_BATCH, normalize_embeddings=True,
                               convert_to_numpy=True)
        top = np.argsort(-(vectors @ corpus_vectors.T), axis=1)[:, :10]
        set_ranks = {}
        for (q, _), row in zip(items, top):
            set_ranks[q["id"]] = next((r + 1 for r, i in enumerate(row) if i in eval_answers[q["id"]]), None)
        ranks[name] = set_ranks
        rows.append({"test set": name, "questions": len(items),
                     f"hit@{TOP_K}": np.mean([r is not None and r <= TOP_K for r in set_ranks.values()]),
                     "MRR@10": np.mean([1 / r if r else 0.0 for r in set_ranks.values()])})
    return pd.DataFrame(rows).set_index("test set"), ranks


t0 = time.time()
if base_vectors is not None:
    corpus_before = base_vectors[eval_idx]
    print("Passage vectors before training: taken from the vector store")
else:
    print("Embedding the test corpus with the base model...")
    corpus_before = embed_corpus()
before, ranks_before = evaluate(corpus_before)
print(f"Done in {time.time() - t0:.0f}s")
before.style.format({f"hit@{TOP_K}": "{:.0%}", "MRR@10": "{:.3f}"})

W1112 21:16:59.426000 22412 torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Test corpus: 9341 passages (230 of them answer a test question)
Embedding the test corpus with the base model...


Batches:   0%|          | 0/292 [00:00<?, ?it/s]

Done in 210s


,questions,hit@6,MRR@10
test set,,,
"Egyptian, as typed",55,80%,0.663
"Egyptian, after the app's dialect rewrite",55,96%,0.800
Standard Arabic (original questions),57,98%,0.901
English,21,81%,0.599


## 5. Fine-tune
First the original model embeds every target text (the "teacher" vectors); then the top layers learn to reproduce
them: each Egyptian text is pulled onto the vector of its standard-Arabic twin, and every "keep" text onto its own
original vector. The distance printed at the end (1 − cosine similarity) shows how close Egyptian and standard
Arabic have come: it should fall clearly, for example from ≈0.2 to well under 0.1.

In [6]:
import contextlib

from tqdm.auto import tqdm

# 1) What the original bge-m3 says: its vectors for every "teacher" text (computed once, before any training)
model.max_seq_length = MAX_SEQ_LENGTH
teacher_texts = list(dict.fromkeys(e["teacher"] for e in examples))
t0 = time.time()
print(f"Embedding {len(teacher_texts)} texts with the original model (the targets)...")
teacher_vectors = torch.from_numpy(model.encode(teacher_texts, batch_size=ENCODE_BATCH, normalize_embeddings=True,
                                                convert_to_numpy=True, show_progress_bar=True)).float().to(DEVICE)
teacher_index = {text: i for i, text in enumerate(teacher_texts)}
print(f"Done in {time.time() - t0:.0f}s")

# 2) Freeze everything, then let the top TRAIN_LAYERS transformer layers learn
layers = model[0].auto_model.encoder.layer
for p in model.parameters():
    p.requires_grad = False
for layer in layers[-TRAIN_LAYERS:]:
    for p in layer.parameters():
        p.requires_grad = True
trainable = [p for p in model.parameters() if p.requires_grad]
print(f"Training the top {TRAIN_LAYERS} of {len(layers)} layers: "
      f"{sum(p.numel() for p in trainable) / 1e6:.0f}M of {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M parameters")

prepare = getattr(model, "preprocess", None) or model.tokenize   # sentence-transformers >= 6 / older


def embed(texts: list[str]) -> torch.Tensor:
    features = prepare(texts)
    features = {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in features.items()}
    return model(features)["sentence_embedding"]


train_rng = random.Random(SEED)
epoch_batches = []
for _ in range(EPOCHS):
    order = examples[:]
    train_rng.shuffle(order)
    epoch_batches.append([order[i:i + BATCH_SIZE] for i in range(0, len(order), BATCH_SIZE)])
total_steps = sum(len(b) for b in epoch_batches)
warmup_steps = max(1, int(total_steps * WARMUP))
optimizer = torch.optim.AdamW(trainable, lr=LEARNING_RATE, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lambda s: min((s + 1) / warmup_steps, max(0.0, (total_steps - s) / max(1, total_steps - warmup_steps))))
use_amp = DEVICE == "cuda"
scaler = torch.amp.GradScaler("cuda") if use_amp else None
autocast = (lambda: torch.autocast("cuda", dtype=torch.float16)) if use_amp else contextlib.nullcontext
print(f"{len(examples)} examples -> {total_steps} steps of {BATCH_SIZE} ({EPOCHS} epoch(s)), mixed precision: {use_amp}")

# 3) Training: pull each student vector onto its teacher vector (1 - cosine similarity)
model.train()
losses, egyptian_losses, t0 = [], [], time.time()
progress = tqdm(total=total_steps, desc="training")
try:
    for epoch, batches in enumerate(epoch_batches, 1):
        for batch in batches:
            targets = teacher_vectors[[teacher_index[e["teacher"]] for e in batch]]
            with autocast():
                student = embed([e["student"] for e in batch])
            distance = 1 - (student.float() * targets).sum(dim=1)
            loss = distance.mean()
            optimizer.zero_grad(set_to_none=True)
            stepped = True
            if scaler:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(trainable, 1.0)
                scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                stepped = scaler.get_scale() >= scale   # fp16: a step whose gradients overflowed is skipped
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(trainable, 1.0)
                optimizer.step()
            if stepped:
                scheduler.step()
            losses.append(loss.item())
            egyptian = [d for d, e in zip(distance.tolist(), batch) if not e["kind"].startswith("keep")]
            egyptian_losses.extend(egyptian)
            progress.update(1)
            progress.set_postfix(loss=f"{np.mean(losses[-50:]):.4f}", epoch=epoch)
except torch.cuda.OutOfMemoryError:
    print("\n⚠️ The GPU ran out of memory. Set BATCH_SIZE = 8 (or MAX_SEQ_LENGTH = 192) in the first cell, "
          "restart the kernel and run again.")
    raise
finally:
    progress.close()
    model.eval()
    model.max_seq_length = FULL_MAX_LENGTH
    for p in model.parameters():
        p.requires_grad = False

n = max(1, min(200, len(egyptian_losses) // 5))
print(f"Trained in {(time.time() - t0) / 60:.1f} min | distance Egyptian -> standard Arabic: "
      f"{np.mean(egyptian_losses[:n]):.3f} at the start, {np.mean(egyptian_losses[-n:]):.3f} at the end (0 = same meaning)")

Embedding 9684 texts with the original model (the targets)...


Batches:   0%|          | 0/303 [00:00<?, ?it/s]

Done in 73s
Training the top 6 of 24 layers: 76M of 568M parameters
11641 examples -> 1456 steps of 16 (2 epoch(s)), mixed precision: True


training:   0%|          | 0/1456 [00:00<?, ?it/s]

Trained in 5.6 min | distance Egyptian -> standard Arabic: 0.177 at the start, 0.072 at the end (0 = same meaning)


## 6. Test after training
Same questions, same passages, re-embedded with the fine-tuned model.

In [7]:
t0 = time.time()
print("Embedding the test corpus with the fine-tuned model...")
corpus_after = embed_corpus()
after, ranks_after = evaluate(corpus_after)
print(f"Done in {time.time() - t0:.0f}s")

comparison = pd.DataFrame({
    "questions": before["questions"],
    f"hit@{TOP_K} before": before[f"hit@{TOP_K}"], f"hit@{TOP_K} after": after[f"hit@{TOP_K}"],
    "MRR@10 before": before["MRR@10"], "MRR@10 after": after["MRR@10"],
})
comparison.style.format({c: "{:.0%}" if "hit" in c else "{:.3f}" for c in comparison.columns if c != "questions"})

Embedding the test corpus with the fine-tuned model...


Batches:   0%|          | 0/292 [00:00<?, ?it/s]

Done in 211s


,questions,hit@6 before,hit@6 after,MRR@10 before,MRR@10 after
test set,,,,,
"Egyptian, as typed",55,80%,80%,0.663,0.665
"Egyptian, after the app's dialect rewrite",55,96%,89%,0.800,0.718
Standard Arabic (original questions),57,98%,96%,0.901,0.867
English,21,81%,71%,0.599,0.613


In [8]:
# Which Egyptian questions changed? (rank of the first passage with the answer, '-' = not in the top 10)
name = "Egyptian, as typed"
fmt = lambda r: str(r) if r else "-"   # noqa: E731
changes = [{"id": qid, "question": next(q["question"] for q, _ in TEST_SETS[name] if q["id"] == qid),
            "rank before": fmt(ranks_before[name][qid]), "rank after": fmt(ranks_after[name][qid])}
           for qid in ranks_before[name] if ranks_before[name][qid] != ranks_after[name][qid]]
better = sum((ranks_after[name][c["id"]] or 99) < (ranks_before[name][c["id"]] or 99) for c in changes)
print(f"{better} Egyptian questions ranked their answer higher after training, {len(changes) - better} lower, "
      f"{len(ranks_before[name]) - len(changes)} unchanged")
pd.DataFrame(changes) if changes else None

10 Egyptian questions ranked their answer higher after training, 8 lower, 37 unchanged


,id,question,rank before,rank after
0,e04,الفدان عايز كام كيلو تقاوي قمح لو بسقي بالبيفوت؟,1,7
1,e05,لو هزرع القمح نقر على مصاطب، أحط كام كيلو تقاوي؟,2,6
2,e19,الفدان في أراضي الوادي ياخد كام كيلو تقاوي كتان؟,1,7
3,e20,التلقيح البكتيري لتقاوي الفول البلدي بيفيد في إيه؟,2,1
4,e24,الفدان عايز كام جرام بذرة أوطة هجين؟,5,1
5,e25,الأوطة عندي بتعفن من تحت عند الطرف، السبب إيه؟,-,10
6,e27,أعمل إيه في البصل بعد ما أقلعه عشان يستحمل التخزين مدة طويلة؟,2,1
7,e29,أقسي شتلات البتنجان إزاي قبل ما أنقلها الأرض المستديمة؟,1,2
8,e30,الخيار بيتقطف بعد كام يوم من الزراعة؟,1,2
9,e32,ليه ساق الكرنب بتطلع مجوفة؟,4,6


## 7. Save
Saved to `models/bge-m3-egyptian/` **only if it is better**: the Egyptian questions as typed rank their answers
higher (MRR up by more than 0.01), and no test set — Egyptian after the app's rewrite, standard Arabic, English —
finds fewer answers in the top 6 or ranks them lower. Otherwise nothing is saved and the app keeps using
`BAAI/bge-m3`. `saqya_finetune.json` in the folder records the settings and the numbers; the table goes to
`notebooks/results/egyptian_finetune_results.md`.

In [9]:
# Saved only when it is better: the Egyptian questions (as typed) rank their answers higher, and no test set —
# Egyptian after the app's rewrite, standard Arabic, English — finds fewer answers or ranks them lower.
gain = after.loc["Egyptian, as typed", "MRR@10"] - before.loc["Egyptian, as typed", "MRR@10"]
worse = [s for s in before.index
         if after.loc[s, f"hit@{TOP_K}"] < before.loc[s, f"hit@{TOP_K}"] or after.loc[s, "MRR@10"] < before.loc[s, "MRR@10"] - 0.005]
better = gain > 0.01 and not worse
metrics = {
    "base_model": BASE_MODEL,
    "trained_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "device": DEVICE,
    "method": "distillation: Egyptian text -> the original model's vector for the same text in standard Arabic",
    "settings": {"train_layers": TRAIN_LAYERS, "epochs": EPOCHS, "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE,
                 "max_seq_length": MAX_SEQ_LENGTH, "examples": len(examples), "steps": total_steps},
    "examples_by_kind": dict(Counter(e["kind"] for e in examples)),
    "test_corpus_passages": len(eval_idx),
    "before": before.round(4).to_dict(orient="index"),
    "after": after.round(4).to_dict(orient="index"),
    "kept": bool(better or not KEEP_ONLY_IF_BETTER),
}
lines = ["| test set | questions | " + " | ".join(c for c in comparison.columns if c != "questions") + " |",
         "|---|---:|" + "---:|" * (len(comparison.columns) - 1)]
for name, row in comparison.iterrows():
    cells = [("{:.0%}" if "hit" in c else "{:.3f}").format(row[c]) for c in comparison.columns if c != "questions"]
    lines.append(f"| {name} | {int(row['questions'])} | " + " | ".join(cells) + " |")
report = ("# Egyptian Arabic fine-tune of bge-m3\n\n"
          f"Distillation, {len(examples)} examples, {total_steps} steps, top {TRAIN_LAYERS} layers, lr {LEARNING_RATE}, {DEVICE}. "
          f"Embeddings-only search over {len(eval_idx)} passages.\n\n" + "\n".join(lines) + "\n")
(RESULTS_DIR / "egyptian_finetune_results.md").write_text(report, encoding="utf-8")

if better or not KEEP_ONLY_IF_BETTER:
    if OUTPUT_DIR.exists():
        shutil.rmtree(OUTPUT_DIR)
    OUTPUT_DIR.parent.mkdir(parents=True, exist_ok=True)
    model.save(str(OUTPUT_DIR))
    (OUTPUT_DIR / "saqya_finetune.json").write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding="utf-8")
    size_gb = sum(f.stat().st_size for f in OUTPUT_DIR.rglob("*") if f.is_file()) / 1024 ** 3
    print(f"✅ Saved to {rel(OUTPUT_DIR)} ({size_gb:.1f} GB). Egyptian questions MRR {gain:+.3f}"
          + (", nothing got worse." if not worse else f" — but worse on: {', '.join(worse)} (KEEP_ONLY_IF_BETTER is off)."))
    print("Next: run rag_pipeline.ipynb (it uses this model automatically), then restart the backend.")
else:
    reason = (f"worse on: {', '.join(worse)}" if worse else f"Egyptian questions improved too little (MRR {gain:+.3f})")
    print(f"❌ Not saved: {reason}. The app keeps using {BASE_MODEL}"
          + (f" (the earlier model in {rel(OUTPUT_DIR)} is left as it is)" if OUTPUT_DIR.exists() else "")
          + ".\n   The app works well without it: the dialect rewrite already handles most Egyptian questions.")
print(f"Results: {rel(RESULTS_DIR / 'egyptian_finetune_results.md')}")

❌ Not saved: worse on: Egyptian, after the app's dialect rewrite, Standard Arabic (original questions), English. The app keeps using BAAI/bge-m3.
   The app works well without it: the dialect rewrite already handles most Egyptian questions.
Results: notebooks/results/egyptian_finetune_results.md


**Next:** run `rag_pipeline.ipynb`. Its first cell prints `Embeddings : models/bge-m3-egyptian` when it found the
new model (set `USE_EGYPTIAN_MODEL = False` there to go back to `BAAI/bge-m3`). It embeds every document with it and
writes the model's path into `backend/data/vector_store/rag_config.json`; restart the backend and it loads the same
model. Uploaded documents and voice questions use it too.

**Docker:** the model folder is not inside `backend/`, so mount it when you run the container:
`docker run -v "$(pwd)/models:/models:ro" ...` (see the README).